# 理财监测 重构

**任务编号**: T43  
**重构日期**: 2026-02-15  

---

## 1. 项目概述

理财监测是一个持续追踪理财市场表现的数据采集和分析系统，主要监控理财产品的业绩表现、规模变化、期限分布以及纯债基金的收益情况。

### 1.1 功能描述

本系统包含四个核心功能模块：

1. **理财业绩跟踪** - 通过同花顺p04838接口获取理财产品业绩数据，计算年化收益率
2. **纯债基金业绩表现** - 从数据库获取纯债基金净值数据，计算滚动收益率
3. **理财规模跟踪** - 爬取证券之星网站的理财规模数据
4. **理财期限跟踪** - 通过同花顺p02186接口获取理财产品期限分布数据

### 1.2 数据源

| 数据源 | 接口/方式 | 主要数据 | 频率 |
|--------|----------|---------|------|
| 同花顺 iFinD | p04838 接口 | 理财产品业绩数据 | 每日 |
| 同花顺 iFinD | p02186 接口 | 理财产品期限分布 | 每日 |
| 证券之星 | 网页爬虫 | 理财规模数据 | 每周 |
| 数据库 | fund.marketinfo + fund.basicinfo | 纯债基金净值数据 | 每日 |

### 1.3 输出结果

- `理财业绩跟踪` 表：各理财公司不同期间的年化收益率
- `理财期限跟踪` 表：理财产品期限分布统计
- `理财规模` 表：理财市场规模数据
- `fund.纯债基金业绩表现` 表：纯债基金近1月/近1年收益率

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import re
from datetime import datetime, timedelta

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text
import pymysql

# 网络请求
import requests
from bs4 import BeautifulSoup

# 同花顺 iFinD (需要安装 iFinDPy)
try:
    from iFinDPy import THS_iFinDLogin, THS_DR
    THS_AVAILABLE = True
except ImportError:
    THS_AVAILABLE = False
    print("警告: iFinDPy 未安装，同花顺相关功能将不可用")

# 配置文件
from config import Config

print("依赖加载完成")

### 2.2 配置参数

In [ ]:
# 加载配置
config = Config()

# 数据库配置
DB_HOST = config.DB_HOST
DB_PORT = config.DB_PORT
DB_USER = config.DB_USER
DB_PASSWORD = config.DB_PASSWORD
DB_NAME = config.DB_NAME

# 同花顺配置
THS_USERNAME = config.THS_USERNAME
THS_PASSWORD = config.THS_PASSWORD

# 爬虫配置
SCALE_URL = config.SCALE_URL

print(f"数据库: {DB_HOST}:{DB_PORT}/{DB_NAME}")
print(f"同花顺账号: {THS_USERNAME}")

## 3. 数据获取

### 3.1 同花顺API连接

In [ ]:
def connect_ths(username: str, password: str) -> bool:
    """
    连接同花顺 iFinD API
    
    Args:
        username: 同花顺账号
        password: 同花顺密码
    
    Returns:
        bool: 连接是否成功
    """
    if not THS_AVAILABLE:
        print("iFinDPy 未安装，无法连接同花顺")
        return False
    
    result = THS_iFinDLogin(username, password)
    if result == 0:
        print("同花顺连接成功")
        return True
    else:
        print(f"同花顺连接失败，错误码: {result}")
        return False

# 连接同花顺
ths_connected = connect_ths(THS_USERNAME, THS_PASSWORD)

### 3.2 数据库连接

In [ ]:
def create_db_engine(host: str, port: int, user: str, password: str, database: str):
    """
    创建数据库连接引擎
    
    Args:
        host: 数据库主机
        port: 端口号
        user: 用户名
        password: 密码
        database: 数据库名
    
    Returns:
        SQLAlchemy Engine 对象
    """
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}'
    engine = sqlalchemy.create_engine(
        connection_string,
        poolclass=sqlalchemy.pool.NullPool,
        pool_recycle=3600
    )
    return engine

# 创建数据库连接
db_engine = create_db_engine(DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME)
db_connection = db_engine.connect()

print("数据库连接成功")

In [ ]:
def create_pymysql_connection(host: str, port: int, user: str, password: str, database: str):
    """
    创建 PyMySQL 连接（用于执行原生SQL）
    
    Returns:
        PyMySQL Connection 对象
    """
    connection = pymysql.connect(
        host=host,
        port=port,
        user=user,
        password=password,
        database=database
    )
    return connection

# 创建 PyMySQL 连接
pymysql_conn = create_pymysql_connection(DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME)

print("PyMySQL 连接成功")

## 4. 数据处理

### 4.1 年化收益率转换

原始数据是期间收益率，需要转换为年化收益率：

- **今年以来收益率**：`年化收益率 = 原始收益率 * (365 / 实际天数)`
- **近1月收益率**：`年化收益率 = 原始收益率 * (365 / 30)`
- **近3月收益率**：`年化收益率 = 原始收益率 * (365 / 90)`
- **近6月收益率**：`年化收益率 = 原始收益率 * (365 / 180)`
- **近1年收益率**：无需转换（已经是年化）

In [ ]:
def calculate_days_since_year_end(current_date: datetime) -> int:
    """
    计算从上一年12月31日到当前日期的天数
    
    Args:
        current_date: 当前日期
    
    Returns:
        int: 天数
    """
    last_year_end = datetime(current_date.year - 1, 12, 31)
    days = (current_date - last_year_end).days
    return days


def annualize_return_rate(df: pd.DataFrame, current_date: datetime) -> pd.DataFrame:
    """
    将期间收益率转换为年化收益率
    
    Args:
        df: 包含原始收益率的数据框
        current_date: 当前日期
    
    Returns:
        转换后的数据框
    """
    # 计算今年以来实际天数
    days = calculate_days_since_year_end(current_date)
    
    # 年化转换
    df['今年以来净值年化增长率'] = df['今年以来净值年化增长率'].astype(float) * 365 / days
    df['近1月净值年化增长率'] = df['近1月净值年化增长率'].astype(float) * 365 / 30
    df['近3月净值年化增长率'] = df['近3月净值年化增长率'].astype(float) * 365 / 90
    df['近6月净值年化增长率'] = df['近6月净值年化增长率'].astype(float) * 365 / 180
    df['近1年净值年化增长率'] = df['近1年净值年化增长率'].astype(float)
    
    return df


# 测试示例
test_date = datetime(2025, 6, 15)
test_days = calculate_days_since_year_end(test_date)
print(f"2025年6月15日距离2024年12月31日: {test_days} 天")

### 4.2 数据清洗

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗数据框，处理缺失值和异常值
    
    Args:
        df: 原始数据框
    
    Returns:
        清洗后的数据框
    """
    # 替换 '--' 为 NaN
    df = df.replace('--', np.nan)
    
    # 替换 pandas NA/NaT/nan 为 None（SQL NULL）
    df = df.replace({pd.NA: None, pd.NaT: None, float('nan'): None})
    
    # 替换 0 为 NaN（对于净值数据）
    df = df.replace(0, np.nan)
    
    return df

print("数据清洗函数定义完成")

## 5. 核心逻辑

### 5.1 理财业绩跟踪

通过同花顺 p04838 接口获取理财产品业绩数据

In [ ]:
def fetch_licaishouyi_data(date_str: str) -> pd.DataFrame:
    """
    从同花顺获取理财业绩数据
    
    Args:
        date_str: 日期字符串，格式 YYYYMMDD
    
    Returns:
        理财业绩数据框
    """
    if not THS_AVAILABLE:
        print("iFinDPy 未安装")
        return None
    
    # 调用同花顺接口
    result = THS_DR(
        'p04838',
        f'edate={date_str};cplx=1',
        'p04838_f001:Y,p04838_f002:Y,p04838_f004:Y,p04838_f005:Y,p04838_f006:Y,p04838_f007:Y,p04838_f008:Y',
        'format:dataframe'
    )
    
    df = result.data if result else None
    return df


def process_licaishouyi_data(df: pd.DataFrame, current_date: datetime) -> pd.DataFrame:
    """
    处理理财业绩数据
    
    Args:
        df: 原始数据框
        current_date: 当前日期
    
    Returns:
        处理后的数据框
    """
    if df is None:
        return None
    
    # 清洗数据
    df = df.replace('--', np.nan)
    
    # 重命名列
    df.columns = [
        '公司名称', 'dt', 
        '今年以来净值年化增长率', '近1月净值年化增长率', 
        '近3月净值年化增长率', '近6月净值年化增长率', 
        '近1年净值年化增长率'
    ]
    
    # 年化转换
    df = annualize_return_rate(df, current_date)
    
    # 处理空值
    df = df.replace({pd.NA: None, pd.NaT: None, float('nan'): None})
    
    return df


def save_licaishouyi_data(df: pd.DataFrame, connection) -> None:
    """
    保存理财业绩数据到数据库（使用 UPSERT）
    
    Args:
        df: 数据框
        connection: 数据库连接
    """
    if df is None or df.empty:
        return
    
    query = """
    INSERT INTO 理财业绩跟踪 (`公司名称`, `dt`, `今年以来净值年化增长率`,
        `近1月净值年化增长率`, `近3月净值年化增长率`, `近6月净值年化增长率`, `近1年净值年化增长率`)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        `今年以来净值年化增长率` = VALUES(`今年以来净值年化增长率`),
        `近1月净值年化增长率` = VALUES(`近1月净值年化增长率`),
        `近3月净值年化增长率` = VALUES(`近3月净值年化增长率`),
        `近6月净值年化增长率` = VALUES(`近6月净值年化增长率`),
        `近1年净值年化增长率` = VALUES(`近1年净值年化增长率`);
    """
    
    with connection.cursor() as cursor:
        for _, row in df.iterrows():
            cursor.execute(query, (
                row['公司名称'], row['dt'], 
                row['今年以来净值年化增长率'], row['近1月净值年化增长率'],
                row['近3月净值年化增长率'], row['近6月净值年化增长率'],
                row['近1年净值年化增长率']
            ))
        connection.commit()


def run_licaishouyi_tracking(start_date: datetime, end_date: datetime, connection) -> None:
    """
    执行理财业绩跟踪（批量处理日期范围）
    
    Args:
        start_date: 开始日期
        end_date: 结束日期
        connection: 数据库连接
    """
    current_date = start_date
    
    while current_date <= end_date:
        date_str = current_date.strftime('%Y%m%d')
        
        # 获取数据
        df = fetch_licaishouyi_data(date_str)
        
        if df is None:
            print(f'{date_str} 未取到数据')
        else:
            # 处理数据
            df = process_licaishouyi_data(df, current_date)
            
            # 保存数据
            save_licaishouyi_data(df, connection)
            
            print(f'{date_str} 处理完成，共 {len(df)} 条记录')
        
        current_date += timedelta(days=1)


print("理财业绩跟踪函数定义完成")

### 5.2 纯债基金业绩计算

从数据库获取纯债基金净值数据，计算滚动收益率

In [ ]:
def fetch_bond_fund_nav(connection, days: int = 368) -> pd.DataFrame:
    """
    获取纯债基金净值数据
    
    Args:
        connection: 数据库连接
        days: 查询天数（默认368天，用于计算近1年收益率）
    
    Returns:
        净值数据框
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    query = f"""
    SELECT
        A.trade_code,
        A.dt,
        A.NAV_ACC
    FROM
        fund.marketinfo A
    INNER JOIN (
        SELECT DISTINCT trade_code
        FROM fund.basicinfo
        WHERE dt = (SELECT MAX(dt) FROM fund.basicinfo)
        AND FUND_INVESTTYPE IN (
            '中长期纯债型基金', 
            '被动指数型债券基金', 
            '短期纯债型基金', 
            '增强指数型债券基金'
        )
    ) D ON A.trade_code = D.trade_code
    WHERE A.dt BETWEEN '{start_date.strftime('%Y-%m-%d')}' AND '{end_date.strftime('%Y-%m-%d')}';
    """
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        data = cursor.fetchall()
    
    df = pd.DataFrame(data, columns=['trade_code', 'dt', 'NAV_ACC'])
    return df


def fill_missing_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    填充缺失日期（前向填充）
    
    Args:
        df: 净值数据框
    
    Returns:
        填充后的数据框
    """
    # 转换日期列
    df['dt'] = pd.to_datetime(df['dt'])
    
    # 替换 0 为 NaN
    df.replace(0, np.nan, inplace=True)
    
    # 按日期填充
    df = df.set_index('dt').groupby('trade_code').apply(
        lambda x: x.resample('D').ffill()
    ).reset_index(level=0, drop=True).reset_index()
    
    return df


def calculate_rolling_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    计算滚动收益率
    
    Args:
        df: 净值数据框
    
    Returns:
        包含滚动收益率的数据框
    """
    # 近1月年化收益率
    df['近1月年化收益率'] = df.groupby('trade_code')['NAV_ACC'].apply(
        lambda x: (x / x.shift(30) - 1) * 365 / 30
    ).reset_index(level=0, drop=True)
    
    # 近1年收益率
    df['近1年收益率'] = df.groupby('trade_code')['NAV_ACC'].apply(
        lambda x: (x / x.shift(365) - 1)
    ).reset_index(level=0, drop=True)
    
    return df


def calculate_average_returns(df: pd.DataFrame, exclude_codes: list = None) -> pd.DataFrame:
    """
    计算每个日期的平均收益率
    
    Args:
        df: 包含滚动收益率的数据框
        exclude_codes: 要排除的基金代码列表
    
    Returns:
        平均收益率数据框
    """
    if exclude_codes:
        df = df[~df['trade_code'].isin(exclude_codes)]
    
    avg_df = df.groupby('dt').agg({
        '近1月年化收益率': 'mean',
        '近1年收益率': 'mean'
    }).reset_index()
    
    avg_df.dropna(inplace=True)
    
    return avg_df


def save_bond_fund_performance(df: pd.DataFrame, connection) -> None:
    """
    保存纯债基金业绩数据
    
    Args:
        df: 平均收益率数据框
        connection: 数据库连接
    """
    query = """
    INSERT INTO fund.纯债基金业绩表现 (dt, 近1月年化收益率, 近1年收益率)
    VALUES (%s, %s, %s)
    ON DUPLICATE KEY UPDATE
        近1月年化收益率 = VALUES(近1月年化收益率),
        近1年收益率 = VALUES(近1年收益率);
    """
    
    with connection.cursor() as cursor:
        for _, row in df.iterrows():
            cursor.execute(query, (
                row['dt'], 
                row['近1月年化收益率'], 
                row['近1年收益率']
            ))
        connection.commit()


def run_bond_fund_performance(connection) -> None:
    """
    执行纯债基金业绩计算
    """
    print("开始获取纯债基金净值数据...")
    df = fetch_bond_fund_nav(connection)
    print(f"获取到 {len(df)} 条净值记录")
    
    print("填充缺失日期...")
    df = fill_missing_dates(df)
    
    print("计算滚动收益率...")
    df = calculate_rolling_returns(df)
    
    print("计算平均收益率...")
    # 排除特定基金（如 511010.OF 国债ETF）
    avg_df = calculate_average_returns(df, exclude_codes=['511010.OF'])
    print(f"计算得到 {len(avg_df)} 天的平均收益率")
    
    print("保存到数据库...")
    save_bond_fund_performance(avg_df, connection)
    print("纯债基金业绩计算完成")


print("纯债基金业绩函数定义完成")

### 5.3 理财规模跟踪

爬取证券之星网站的理财规模数据

In [ ]:
def fetch_webpage_content(url: str) -> BeautifulSoup:
    """
    获取网页内容
    
    Args:
        url: 网页URL
    
    Returns:
        BeautifulSoup 对象
    """
    response = requests.get(url)
    
    if response.status_code == 200:
        response.encoding = response.apparent_encoding
        return BeautifulSoup(response.text, 'html.parser')
    else:
        print(f"无法获取网页内容，状态码: {response.status_code}")
        return None


def extract_scale_data(soup: BeautifulSoup):
    """
    从网页中提取理财规模数据
    
    Args:
        soup: BeautifulSoup 对象
    
    Returns:
        tuple: (规模数字, 日期)
    """
    text = soup.prettify()
    
    # 提取包含规模信息的文本
    pattern = re.compile(r'(截止.*?理财.*?规模.*?万亿)')
    match = pattern.search(text)
    
    if not match:
        print("未找到包含规模信息的文本")
        return None, None
    
    # 提取子串
    start_pos = match.start()
    end_pos = start_pos + 50
    extracted_text = text[start_pos:end_pos]
    print(f"提取的文本: {extracted_text}")
    
    # 提取规模数字
    number_pattern = re.compile(r'规模(.*?)万亿')
    number_match = number_pattern.search(extracted_text)
    
    if number_match:
        potential_numbers = re.findall(r'[\d,\.]+', number_match.group(1))
        number = potential_numbers[0] if potential_numbers else None
    else:
        number = None
    
    # 提取日期
    date_pattern = re.compile(r'(\d{4}年\d{1,2}月\d{1,2}日)')
    date_match = date_pattern.search(extracted_text)
    
    if date_match:
        date_str = date_match.group(1)
        try:
            date_obj = datetime.strptime(date_str, "%Y年%m月%d日")
            formatted_date = date_obj.strftime("%Y-%m-%d")
        except ValueError:
            formatted_date = None
    else:
        formatted_date = None
    
    return number, formatted_date


def get_latest_article_link(soup: BeautifulSoup, base_url: str) -> str:
    """
    获取最新文章链接
    
    Args:
        soup: BeautifulSoup 对象
        base_url: 基础URL
    
    Returns:
        文章链接
    """
    zhangjing_div = soup.find('div', class_='name', title='张菁的文章')
    
    if zhangjing_div:
        first_a_tag = zhangjing_div.find_next('a')
        if first_a_tag and 'href' in first_a_tag.attrs:
            return first_a_tag['href']
    
    return None


def save_scale_data(scale: str, date: str, engine) -> None:
    """
    保存理财规模数据
    
    Args:
        scale: 规模数字
        date: 日期
        engine: SQLAlchemy 引擎
    """
    data = {'dt': [date], '理财规模': [scale]}
    df = pd.DataFrame(data)
    
    try:
        with engine.connect() as conn:
            df.to_sql('理财规模', con=conn, if_exists='append', index=False)
        print(f"理财规模数据已保存: {date}, {scale}万亿")
    except Exception as e:
        print(f"保存失败: {e}")


def run_scale_tracking(url: str, engine) -> None:
    """
    执行理财规模跟踪
    
    Args:
        url: 目标URL
        engine: SQLAlchemy 引擎
    """
    print(f"访问: {url}")
    soup = fetch_webpage_content(url)
    
    if soup is None:
        return
    
    # 获取最新文章链接
    article_link = get_latest_article_link(soup, url)
    
    if article_link:
        print(f"最新文章链接: {article_link}")
        article_soup = fetch_webpage_content(article_link)
        
        if article_soup:
            scale, date = extract_scale_data(article_soup)
            
            if scale and date:
                print(f"提取的规模: {scale}万亿, 日期: {date}")
                save_scale_data(scale, date, engine)
            else:
                print("未能提取规模数据")
    else:
        print("未找到最新文章链接")


print("理财规模跟踪函数定义完成")

### 5.4 理财期限跟踪

通过同花顺 p02186 接口获取理财产品期限分布数据

In [ ]:
def fetch_licaixian_data(date_str: str) -> pd.DataFrame:
    """
    从同花顺获取理财期限数据
    
    Args:
        date_str: 日期字符串，格式 YYYYMMDD
    
    Returns:
        理财期限数据框
    """
    if not THS_AVAILABLE:
        print("iFinDPy 未安装")
        return None
    
    # 调用同花顺接口
    result = THS_DR(
        'p02186',
        f'type=收益起始日期;sdate={date_str};edate={date_str}',
        'p02186_f002:Y,p02186_f003:Y,p02186_f005:Y,p02186_f006:Y,p02186_f008:Y,p02186_f009:Y,'
        'p02186_f011:Y,p02186_f012:Y,p02186_f014:Y,p02186_f015:Y,p02186_f017:Y,p02186_f018:Y,'
        'p02186_f020:Y,p02186_f021:Y,p02186_f023:Y,p02186_f024:Y',
        'format:dataframe'
    )
    
    df = result.data if result else None
    return df


def process_licaixian_data(df: pd.DataFrame, date_str: str) -> pd.DataFrame:
    """
    处理理财期限数据
    
    Args:
        df: 原始数据框
        date_str: 日期字符串（YYYY-MM-DD格式）
    
    Returns:
        处理后的数据框
    """
    if df is None:
        return None
    
    # 筛选合计行
    df = df[df['p02186_f002'] == '合计']
    
    if df.empty:
        return None
    
    # 替换日期
    df['p02186_f002'].iloc[0] = date_str
    
    # 选择需要的列
    df = df[['p02186_f002', 'p02186_f003', 'p02186_f005', 'p02186_f006', 
             'p02186_f008', 'p02186_f009', 'p02186_f011', 'p02186_f012', 
             'p02186_f014', 'p02186_f015', 'p02186_f017', 'p02186_f018', 
             'p02186_f020', 'p02186_f021', 'p02186_f023', 'p02186_f024']]
    
    # 重命名列
    df.columns = [
        'dt', '1个月以内数量', '1个月以内占比',
        '1-3月数量', '1-3月占比',
        '3-6月数量', '3-6月占比',
        '6-12月数量', '6-12月占比',
        '12-24月数量', '12-24月占比',
        '24月以上数量', '24月以上占比',
        '未公布数量', '未公布占比', '总数'
    ]
    
    return df


def save_licaixian_data(df: pd.DataFrame, engine) -> None:
    """
    保存理财期限数据
    
    Args:
        df: 数据框
        engine: SQLAlchemy 引擎
    """
    if df is None or df.empty:
        return
    
    try:
        with engine.connect() as conn:
            df.to_sql('理财期限跟踪', con=conn, if_exists='append', index=False)
            conn.commit()
        print(f"理财期限数据已保存")
    except Exception as e:
        print(f"保存失败: {e}")


def run_licaixian_tracking(engine) -> None:
    """
    执行理财期限跟踪
    """
    # 获取当前日期（前一天的）
    current_date = datetime.now() - timedelta(days=1)
    date_str = current_date.strftime('%Y%m%d')
    dt_str = current_date.strftime('%Y-%m-%d')
    
    print(f"获取 {dt_str} 的理财期限数据...")
    
    # 获取数据
    df = fetch_licaixian_data(date_str)
    
    if df is None:
        print(f'{dt_str} 未取到数据')
        return
    
    # 处理数据
    df = process_licaixian_data(df, dt_str)
    
    # 保存数据
    save_licaixian_data(df, engine)


print("理财期限跟踪函数定义完成")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def main():
    """
    主函数 - 执行所有监测任务
    """
    print("="*50)
    print("理财监测系统启动")
    print("="*50)
    
    # 1. 理财业绩跟踪（如果需要回溯历史数据）
    # start_date = datetime(2024, 1, 1)
    # end_date = datetime.now()
    # run_licaishouyi_tracking(start_date, end_date, pymysql_conn)
    
    # 2. 纯债基金业绩计算
    print("\n" + "-"*50)
    print("执行: 纯债基金业绩计算")
    print("-"*50)
    try:
        run_bond_fund_performance(pymysql_conn)
    except Exception as e:
        print(f"纯债基金业绩计算失败: {e}")
    
    # 3. 理财规模跟踪
    print("\n" + "-"*50)
    print("执行: 理财规模跟踪")
    print("-"*50)
    try:
        run_scale_tracking(SCALE_URL, db_engine)
    except Exception as e:
        print(f"理财规模跟踪失败: {e}")
    
    # 4. 理财期限跟踪
    print("\n" + "-"*50)
    print("执行: 理财期限跟踪")
    print("-"*50)
    try:
        run_licaixian_tracking(db_engine)
    except Exception as e:
        print(f"理财期限跟踪失败: {e}")
    
    print("\n" + "="*50)
    print("理财监测系统执行完成")
    print("="*50)


# 执行主函数（取消注释以运行）
# main()

### 6.2 单独执行各模块

In [ ]:
# 单独执行：纯债基金业绩计算
# run_bond_fund_performance(pymysql_conn)

In [ ]:
# 单独执行：理财规模跟踪
# run_scale_tracking(SCALE_URL, db_engine)

In [ ]:
# 单独执行：理财期限跟踪
# run_licaixian_tracking(db_engine)

In [ ]:
# 单独执行：理财业绩跟踪（指定日期范围）
# start_date = datetime(2025, 1, 1)
# end_date = datetime.now()
# run_licaishouyi_tracking(start_date, end_date, pymysql_conn)

### 6.3 结果验证

In [ ]:
def verify_data(table_name: str, engine, limit: int = 5):
    """
    验证数据库中的数据
    
    Args:
        table_name: 表名
        engine: SQLAlchemy 引擎
        limit: 返回记录数
    """
    query = f"SELECT * FROM {table_name} ORDER BY dt DESC LIMIT {limit}"
    
    try:
        df = pd.read_sql(query, engine)
        print(f"\n{table_name} 表最新 {limit} 条记录:")
        print(df)
        return df
    except Exception as e:
        print(f"查询失败: {e}")
        return None

# 验证各表数据（取消注释以运行）
# verify_data('理财业绩跟踪', db_engine)
# verify_data('理财期限跟踪', db_engine)
# verify_data('理财规模', db_engine)
# verify_data('fund.纯债基金业绩表现', db_engine)

## 7. 工具函数（可复用）

In [ ]:
def get_trading_days(start_date: datetime, end_date: datetime) -> list:
    """
    获取交易日列表（排除周末）
    
    Args:
        start_date: 开始日期
        end_date: 结束日期
    
    Returns:
        交易日列表
    """
    trading_days = []
    current = start_date
    
    while current <= end_date:
        if current.weekday() < 5:  # 周一到周五
            trading_days.append(current)
        current += timedelta(days=1)
    
    return trading_days


def safe_float_convert(value, default=None):
    """
    安全转换为浮点数
    
    Args:
        value: 要转换的值
        default: 转换失败时的默认值
    
    Returns:
        浮点数或默认值
    """
    try:
        return float(value)
    except (ValueError, TypeError):
        return default


def batch_insert(cursor, query: str, data_list: list, batch_size: int = 1000):
    """
    批量插入数据
    
    Args:
        cursor: 数据库游标
        query: SQL语句
        data_list: 数据列表
        batch_size: 批次大小
    """
    for i in range(0, len(data_list), batch_size):
        batch = data_list[i:i+batch_size]
        cursor.executemany(query, batch)
    

print("工具函数定义完成")

In [ ]:
# 关闭连接
def cleanup():
    """
    清理资源
    """
    try:
        db_connection.close()
        print("SQLAlchemy 连接已关闭")
    except:
        pass
    
    try:
        pymysql_conn.close()
        print("PyMySQL 连接已关闭")
    except:
        pass


# cleanup()